# **Practice: Quantization**

## **Environment**

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.ao.quantization import quantize_dynamic
import os, time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## **(1) Select a small pre-trained sentiment analysis model from Hugging Face.**

In [ ]:
model_id = "distilbert-base-uncased-finetuned-sst-2-english"

# Load tokenizer and FP32 model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_fp32 = AutoModelForSequenceClassification.from_pretrained(model_id)
model_fp32.eval()  # set to evaluation mode

# Example input text
text = "The user interface is beautiful, but the app is very slow."
inputs = tokenizer(text, return_tensors="pt")  # text → input_ids, attention_mask

## **(2) Run baseline inference with the original FP32 model.**


In [ ]:
# Run inference with FP32 model
with torch.no_grad():
    start = time.time()
    outputs_fp32 = model_fp32(**inputs)
    end = time.time()

logits_fp32 = outputs_fp32.logits
probs_fp32 = torch.softmax(logits_fp32, dim=-1)[0]
pred_label_fp32 = torch.argmax(probs_fp32).item()

print("FP32 prediction:", pred_label_fp32, probs_fp32.tolist())
print(f"FP32 inference time: {end - start:.4f} seconds")

## **(3) Quantize Model and Compare Results**

In [ ]:
# Apply dynamic quantization to the FP32 model.
# This will convert specified layers (here, all nn.Linear layers) from FP32 weights to INT8 weights.
model_int8 = quantize_dynamic(
    model_fp32,        # original full-precision (FP32) model
    {nn.Linear},       # set of layer types to quantize: all Linear layers in the model
    dtype=torch.qint8  # target quantized data type for weights (8-bit signed integers)
)

# Switch the quantized model to evaluation mode.
# This disables dropout and other training-specific behavior for correct inference.
model_int8.eval()

In [ ]:
with torch.no_grad():                     # Disable gradient calculation for faster inference
    start = time.time()                   # Record start time
    outputs_int8 = model_int8(**inputs)   # Run forward pass with the quantized INT8 model
    end = time.time()                     # Record end time

logits_int8 = outputs_int8.logits         # Extract raw output scores (logits)
probs_int8 = torch.softmax(logits_int8, dim=-1)[0]  # Convert logits to probability distribution
pred_label_int8 = torch.argmax(probs_int8).item()   # Get the predicted class index

print("INT8 prediction:", pred_label_int8, probs_int8.tolist())
print(f"INT8 inference time: {end - start:.4f} seconds")

In [ ]:
torch.save(model_fp32.cpu().state_dict(), "model_fp32.pt") # # Save the original FP32 model weights
torch.save(model_int8.state_dict(), "model_int8.pt") # Save the quantized INT8 model weights

# Get file sizes in megabytes (bytes → MB)
size_fp32 = os.path.getsize("model_fp32.pt") / (1024 * 1024)
size_int8 = os.path.getsize("model_int8.pt") / (1024 * 1024)

# Print the size of each model
print(f"FP32 model size: {size_fp32:.2f} MB")
print(f"INT8 model size: {size_int8:.2f} MB")

# Compute and print percentage of size reduction
print(f"Size reduction: {100 * (1 - size_int8 / size_fp32):.1f}%")